# 01 — Data Exploration
Look at a raw Sentinel-2 scene, ground sensor readings, and labels for one field before building the preprocessing pipeline.

In [ ]:
import sys, os, glob
sys.path.append(os.path.abspath("../src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

RAW_DIR = "../data/raw"
FIELD_ID = "field_001"  # change to whichever field id exists under data/raw/sentinel2/


## Inventory raw files

In [ ]:
scene_paths = sorted(glob.glob(os.path.join(RAW_DIR, "sentinel2", FIELD_ID, "*.tif")))
sensor_path = os.path.join(RAW_DIR, "sensors", f"{FIELD_ID}.csv")
label_path = os.path.join("../data/labels", f"{FIELD_ID}.csv")

print(f"{len(scene_paths)} Sentinel-2 scenes found")
for p in scene_paths[:5]:
    print(" ", p)
print("sensor file exists:", os.path.exists(sensor_path))
print("label file exists:", os.path.exists(label_path))


## Load and inspect one Sentinel-2 scene

In [ ]:
with rasterio.open(scene_paths[0]) as src:
    scene = src.read().astype(np.float32)  # (C, H, W)
    print("shape:", scene.shape, "dtype:", scene.dtype)
    print("crs:", src.crs)
    print("bounds:", src.bounds)


## Plot an RGB composite (bands assumed [B02,B03,B04,B08,B11,B12] -> RGB = idx 2,1,0)

In [ ]:
def to_rgb(scene, r=2, g=1, b=0, clip=3000):
    rgb = np.stack([scene[r], scene[g], scene[b]], axis=-1)
    rgb = np.clip(rgb, 0, clip) / clip
    return rgb

plt.figure(figsize=(6, 6))
plt.imshow(to_rgb(scene))
plt.title(f"{FIELD_ID} — RGB composite")
plt.axis("off")
plt.show()


## Quick NDVI look

In [ ]:
from features import ndvi

ndvi_map = ndvi(scene)
plt.figure(figsize=(6, 6))
im = plt.imshow(ndvi_map, cmap="RdYlGn", vmin=-1, vmax=1)
plt.title("NDVI")
plt.axis("off")
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.show()

print("NDVI stats — mean: %.3f, std: %.3f" % (np.nanmean(ndvi_map), np.nanstd(ndvi_map)))


## Sensor data

In [ ]:
sensor_df = pd.read_csv(sensor_path, parse_dates=["timestamp"])
display(sensor_df.describe())
sensor_df.head()


In [ ]:
fig, axes = plt.subplots(len(sensor_df.columns) - 1, 1, figsize=(10, 2.5 * (len(sensor_df.columns) - 1)), sharex=True)
for ax, col in zip(axes, [c for c in sensor_df.columns if c != "timestamp"]):
    ax.plot(sensor_df["timestamp"], sensor_df[col])
    ax.set_ylabel(col)
plt.xlabel("time")
plt.tight_layout()
plt.show()


## Labels

In [ ]:
labels_df = pd.read_csv(label_path, parse_dates=["timestamp"])
display(labels_df.describe())

fig, ax = plt.subplots(figsize=(10, 4))
for col in ["stress_risk", "water_risk", "pest_risk"]:
    ax.plot(labels_df["timestamp"], labels_df[col], label=col, marker="o")
ax.legend()
ax.set_ylim(0, 1)
ax.set_title(f"{FIELD_ID} — risk labels over time")
plt.show()


### Notes
- Confirm band order matches `SENTINEL2_BANDS` in `src/preprocessing.py` before proceeding.
- Check for cloud cover / missing scenes — gaps will need interpolation or exclusion in `02_preprocessing.ipynb`.
- Check sensor sampling frequency vs. Sentinel-2 revisit time (~5 days) to plan alignment/interpolation.